# NUS ST5201X — Statistical Foundations of Data Science
## Detailed Bokeh case study: Wisconsin Diagnostic Breast Cancer data

This notebook is an expanded, **Bokeh-first** companion to ST5201X.  It is designed to be read like a guided practical rather than a collection of disconnected snippets.

We use a single real dataset throughout and repeatedly return to the same questions:

> **What is random? What is the population quantity? What is the estimator? How uncertain is it? What assumptions are being made? What happens when those assumptions or the sample size change?**

### Case-study outcome

We recode diagnosis as

$$
Y=\begin{cases}
1,&\text{malignant}\\
0,&\text{benign}
\end{cases}
$$

and use continuous cell-nucleus measurements such as radius, texture, perimeter and area.

### Coverage

The notebook includes:

1. exploratory analysis and robust summaries;
2. random variables and Bernoulli/binomial modelling;
3. joint, marginal and conditional distributions;
4. covariance and correlation;
5. ECDF and kernel density estimation;
6. bandwidth-sensitivity experiments;
7. Law of Large Numbers simulations;
8. Central Limit Theorem scenarios, including skewed data;
9. Method of Moments and Maximum Likelihood;
10. estimator bias/variance/consistency simulation;
11. bootstrap uncertainty for means, medians and group differences;
12. classical and bootstrap confidence intervals;
13. Welch, Mann–Whitney and permutation tests;
14. effect sizes, Type-I/II error and power;
15. Bayesian Beta–Bernoulli updating and prior sensitivity;
16. linear regression with diagnostics;
17. a synthetic assumption-stress scenario;
18. logistic regression, odds ratios and likelihood-ratio testing;
19. threshold tuning, ROC and calibration with Bokeh;
20. statistical significance vs predictive usefulness;
21. reusable OOP utilities and computational-complexity notes.

---

## The central reasoning chain

$$
\boxed{
\text{Population}
\rightarrow
\text{Probability model}
\rightarrow
\text{Sample}
\rightarrow
\text{Estimator}
\rightarrow
\text{Sampling distribution}
\rightarrow
\text{Uncertainty}
\rightarrow
\text{Inference}
\rightarrow
\text{Prediction}
}
$$

# 1. Environment and reproducibility

The dataset is bundled with scikit-learn, so the notebook does not depend on a network connection.

Bokeh is used for **all analytical visualisations**.  The helper functions below deliberately avoid version-fragile settings such as `legend.location = "best"`; all legend locations are explicit Bokeh-supported values.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Iterable, Mapping

import numpy as np
import pandas as pd
from scipy import stats

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    roc_curve,
    confusion_matrix,
)

import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import row, column
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    LinearColorMapper,
    ColorBar,
    BasicTicker,
    DataTable,
    TableColumn,
    NumberFormatter,
    Div,
)
from bokeh.palettes import Viridis256

output_notebook(hide_banner=True)

RANDOM_STATE = 42
RNG = np.random.default_rng(RANDOM_STATE)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1.1 Robust Bokeh helper layer

Repeated plotting code obscures statistical reasoning.  We therefore build a small reusable plotting layer.

Two implementation details are intentional:

* every DataFrame column is converted to a unique string before creating a `ColumnDataSource`;
* legends use locations such as `top_left` or `top_right`, never the unsupported value `best`.

This keeps the notebook compatible with modern Bokeh 3.x.

In [2]:
def _clean_frame(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy().reset_index(drop=True)
    x.columns = [str(c) for c in x.columns]
    if len(set(x.columns)) != len(x.columns):
        seen = {}
        cols = []
        for c in x.columns:
            seen[c] = seen.get(c, 0) + 1
            cols.append(c if seen[c] == 1 else f'{c}_{seen[c]}')
        x.columns = cols
    return x


def show_table(df: pd.DataFrame, title: str, width: int = 900, height: int = 260):
    x = _clean_frame(df)
    source = ColumnDataSource(x)
    columns = []
    for c in x.columns:
        if pd.api.types.is_numeric_dtype(x[c]):
            fmt = NumberFormatter(format='0,0.0000')
            columns.append(TableColumn(field=c, title=c.replace('_', ' ').title(), formatter=fmt))
        else:
            columns.append(TableColumn(field=c, title=c.replace('_', ' ').title()))
    table = DataTable(source=source, columns=columns, width=width, height=height, index_position=None)
    show(column(Div(text=f'<h3>{title}</h3>'), table))


def histogram_plot(values, title, x_label, bins=30, density=False, width=760, height=420):
    values = np.asarray(values, dtype=float)
    hist, edges = np.histogram(values, bins=bins, density=density)
    p = figure(width=width, height=height, title=title, tools='pan,wheel_zoom,box_zoom,reset,save')
    p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], alpha=0.65)
    p.xaxis.axis_label = x_label
    p.yaxis.axis_label = 'Density' if density else 'Count'
    return p


def ecdf_plot(values, title, x_label, width=760, height=420):
    x = np.sort(np.asarray(values, dtype=float))
    y = np.arange(1, len(x)+1) / len(x)
    p = figure(width=width, height=height, title=title, tools='pan,wheel_zoom,box_zoom,reset,save')
    p.step(x, y, mode='after', line_width=2)
    p.xaxis.axis_label = x_label
    p.yaxis.axis_label = 'Empirical CDF'
    return p


def multi_line_plot(df, x, series: Mapping[str, str], title, x_label, y_label, width=780, height=430):
    p = figure(width=width, height=height, title=title, tools='pan,wheel_zoom,box_zoom,reset,save')
    for col, label in series.items():
        p.line(df[x], df[col], line_width=2, legend_label=label)
        p.scatter(df[x], df[col], size=6)
    p.xaxis.axis_label = x_label
    p.yaxis.axis_label = y_label
    if p.legend:
        p.legend.location = 'top_right'
        p.legend.click_policy = 'hide'
    return p


def correlation_heatmap(corr: pd.DataFrame, title='Correlation matrix', width=700, height=620):
    labels = list(corr.columns)
    long = corr.rename_axis('y').reset_index().melt(id_vars='y', var_name='x', value_name='corr')
    mapper = LinearColorMapper(palette=Viridis256, low=-1, high=1)
    source = ColumnDataSource(long)
    p = figure(
        x_range=labels,
        y_range=list(reversed(labels)),
        width=width,
        height=height,
        title=title,
        tools='hover,save,reset',
        tooltips=[('pair', '@x × @y'), ('correlation', '@corr{0.000}')],
        toolbar_location='above',
    )
    p.rect(x='x', y='y', width=1, height=1, source=source,
           fill_color={'field':'corr', 'transform':mapper}, line_color=None)
    p.xaxis.major_label_orientation = 0.9
    p.add_layout(ColorBar(color_mapper=mapper, ticker=BasicTicker()), 'right')
    return p

# 2. Load and understand the dataset

The original scikit-learn coding is `0 = malignant, 1 = benign`.  We invert it so that the Bernoulli event $Y=1$ means malignant.

In [3]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()
df.columns = [c.replace(' ', '_') for c in df.columns]
df['malignant'] = (df['target'] == 0).astype(int)
df = df.drop(columns='target')
feature_cols = [c for c in df.columns if c != 'malignant']

print(f'Rows: {len(df):,}')
print(f'Predictors: {len(feature_cols)}')
print(f'Missing values: {df.isna().sum().sum()}')
display(df.head())

Rows: 569
Predictors: 30
Missing values: 0


,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,malignant
0,17.9900,10.3800,122.8000,"1,001.0000",0.1184,0.2776,0.3001,0.1471,0.2419,0.0787,1.0950,0.9053,8.5890,153.4000,0.0064,0.0490,0.0537,0.0159,0.0300,0.0062,25.3800,17.3300,184.6000,"2,019.0000",0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,1
1,20.5700,17.7700,132.9000,"1,326.0000",0.0847,0.0786,0.0869,0.0702,0.1812,0.0567,0.5435,0.7339,3.3980,74.0800,0.0052,0.0131,0.0186,0.0134,0.0139,0.0035,24.9900,23.4100,158.8000,"1,956.0000",0.1238,0.1866,0.2416,0.1860,0.2750,0.0890,1
2,19.6900,21.2500,130.0000,"1,203.0000",0.1096,0.1599,0.1974,0.1279,0.2069,0.0600,0.7456,0.7869,4.5850,94.0300,0.0062,0.0401,0.0383,0.0206,0.0225,0.0046,23.5700,25.5300,152.5000,"1,709.0000",0.1444,0.4245,0.4504,0.2430,0.3613,0.0876,1
3,11.4200,20.3800,77.5800,386.1000,0.1425,0.2839,0.2414,0.1052,0.2597,0.0974,0.4956,1.1560,3.4450,27.2300,0.0091,0.0746,0.0566,0.0187,0.0596,0.0092,14.9100,26.5000,98.8700,567.7000,0.2098,0.8663,0.6869,0.2575,0.6638,0.1730,1
4,20.2900,14.3400,135.1000,"1,297.0000",0.1003,0.1328,0.1980,0.1043,0.1809,0.0588,0.7572,0.7813,5.4380,94.4400,0.0115,0.0246,0.0569,0.0188,0.0176,0.0051,22.5400,16.6700,152.2000,"1,575.0000",0.1374,0.2050,0.4000,0.1625,0.2364,0.0768,1


Each row is an observed realization. Before observation, quantities such as radius and diagnosis are random variables; after observation they become fixed data values.

This sounds simple, but it is the bridge from a spreadsheet to probability theory.

In [42]:
summary_cols = ['mean_radius','mean_texture','mean_perimeter','mean_area','mean_smoothness','malignant']
summary = df[summary_cols].describe().T.reset_index().rename(columns={'index':'variable'})
show_table(summary, 'Core descriptive statistics', height=280)

# 3. Exploratory data analysis

## 3.1 Mean, median, spread and skewness

For a sample $X_1,\ldots,X_n$,

$$
\bar X=\frac1n\sum_{i=1}^nX_i
$$

is sensitive to extreme observations, while the median is much more robust.

We therefore compare multiple summaries rather than relying on a single number.

In [43]:
eda = pd.DataFrame({
    'variable': feature_cols,
    'mean': [df[c].mean() for c in feature_cols],
    'median': [df[c].median() for c in feature_cols],
    'std': [df[c].std(ddof=1) for c in feature_cols],
    'iqr': [df[c].quantile(.75)-df[c].quantile(.25) for c in feature_cols],
    'skewness': [df[c].skew() for c in feature_cols],
})
show_table(eda.query("variable in ['mean_radius','mean_texture','mean_perimeter','mean_area','mean_smoothness','worst_area']"),
           'Location, spread and skewness', height=280)

In [44]:
p1 = histogram_plot(df['mean_radius'], 'Distribution of mean radius', 'Mean radius', bins=30)
p2 = histogram_plot(df['worst_area'], 'A more strongly skewed feature: worst area', 'Worst area', bins=30)
show(row(p1, p2))

### Scenario A — Why shape matters

`mean_radius` is moderately skewed, whereas `worst_area` is more asymmetric with a longer right tail.

That matters because:

* mean-based procedures are sensitive to tails;
* finite-sample normal approximations can be poorer for strongly skewed distributions;
* robust estimators such as the median can behave differently from the mean;
* transformations may sometimes improve modelling behaviour.

# 4. Bernoulli and binomial probability models

Let

$$
Y_i\sim\operatorname{Bernoulli}(p)
$$

where $Y_i=1$ denotes malignancy.

Then

$$
E[Y_i]=p,\qquad \operatorname{Var}(Y_i)=p(1-p).
$$

If $S=\sum_iY_i$, then

$$
S\sim\operatorname{Binomial}(n,p).
$$

In [45]:
y = df['malignant'].to_numpy()
n = len(y)
s = int(y.sum())
p_hat = y.mean()

pd.DataFrame({
    'quantity':['sample size n','malignant count s','sample proportion p_hat'],
    'value':[n,s,p_hat]
})

,quantity,value
0,sample size n,569.0000
1,malignant count s,212.0000
2,sample proportion p_hat,0.3726


The estimator

$$
\hat p=\frac{S}{n}
$$

is simultaneously:

* the sample mean of the Bernoulli observations;
* the Method-of-Moments estimator;
* the Maximum-Likelihood estimator.

## 4.1 Likelihood profile

For the observed data,

$$
L(p)=p^s(1-p)^{n-s}
$$

and

$$
\ell(p)=s\log p+(n-s)\log(1-p).
$$

The MLE is the parameter value at the peak of this curve.

In [46]:
p_grid = np.linspace(0.05,0.75,700)
ll = s*np.log(p_grid)+(n-s)*np.log(1-p_grid)

p = figure(width=800,height=430,title='Bernoulli log-likelihood profile', tools='pan,wheel_zoom,box_zoom,reset,save')
p.line(p_grid,ll,line_width=3,legend_label='log-likelihood')
p.segment(x0=[p_hat],y0=[ll.min()],x1=[p_hat],y1=[ll.max()],line_dash='dashed',legend_label=f'MLE = {p_hat:.3f}')
p.xaxis.axis_label='Candidate p'
p.yaxis.axis_label='log L(p)'
p.legend.location='bottom_right'
show(p)

# 5. Marginal versus conditional probability

The overall malignancy probability estimates

$$
P(Y=1).
$$

Prediction asks for a conditional probability such as

$$
P(Y=1\mid X=x).
$$

To make the distinction tangible, split mean radius into quartiles and estimate malignancy probability within each quartile.

In [47]:
df['radius_quartile'] = pd.qcut(df['mean_radius'],4,labels=['Q1 smallest','Q2','Q3','Q4 largest'])
conditional = (df.groupby('radius_quartile',observed=True)['malignant']
                 .agg(count='size', malignant_count='sum', conditional_probability='mean')
                 .reset_index())
show_table(conditional,'Estimated P(malignant | radius quartile)',height=230)

p = figure(x_range=conditional['radius_quartile'].astype(str).tolist(), width=760,height=420,
           title='Conditional malignancy probability by mean-radius quartile', tools='hover,save,reset',
           tooltips=[('quartile','@x'),('probability','@top{0.000}')])
p.vbar(x=conditional['radius_quartile'].astype(str), top=conditional['conditional_probability'], width=.75)
p.y_range.start=0
p.y_range.end=1
p.yaxis.axis_label='Estimated probability of malignancy'
show(p)

The probability changes sharply across radius groups.  This is an empirical hint that radius contains predictive information.

But remember: discretizing a continuous predictor is only for illustration; it throws away information. Logistic regression later uses the continuous values directly.

# 6. Covariance and correlation

Correlation measures standardized linear association:

$$
\rho_{XZ}=\frac{\operatorname{Cov}(X,Z)}{\sigma_X\sigma_Z}.
$$

High correlation does **not** imply causation.  It can also create multicollinearity in regression models.

In [10]:
corr_cols=['mean_radius','mean_texture','mean_perimeter','mean_area','mean_smoothness','malignant']
corr=df[corr_cols].corr()
show(correlation_heatmap(corr,'Selected-feature correlation matrix'))

`mean_radius`, `mean_perimeter` and `mean_area` are strongly correlated because they describe related geometry.  If all are entered into the same regression model, coefficient uncertainty can increase even when predictive accuracy remains high.

# 7. Empirical distribution function

The ECDF

$$
\hat F_n(x)=\frac1n\sum_{i=1}^nI(X_i\le x)
$$

requires no parametric distributional assumption.

In [48]:
show(ecdf_plot(df['mean_radius'],'ECDF of mean radius','Mean radius'))

for x0 in [10,15,20]:
    print(f'F_hat({x0}) = {(df["mean_radius"]<=x0).mean():.4f}')

F_hat(10) = 0.0826
F_hat(15) = 0.6960
F_hat(20) = 0.9209


# 8. Kernel density estimation and bandwidth sensitivity

A kernel density estimator is

$$
\hat f_h(x)=\frac1{nh}\sum_{i=1}^nK\left(\frac{x-X_i}{h}\right).
$$

The bandwidth $h$ is a smoothing hyperparameter:

* too small $\rightarrow$ noisy, high-variance estimate;
* too large $\rightarrow$ oversmoothed, high-bias estimate.

This is an early example of the same bias–variance tension encountered throughout machine learning.

In [53]:
values=df['mean_radius'].to_numpy()
x_grid=np.linspace(values.min()-1, values.max()+1, 500)

p=figure(width=800,height=430,title='KDE bandwidth sensitivity',tools='pan,wheel_zoom,box_zoom,reset,save')
for factor,label,color in [(0.35,'narrow bandwidth','red'),(1.0,'default bandwidth','blue'),(2.5,'wide bandwidth','indigo')]:
    kde=stats.gaussian_kde(values)
    kde.set_bandwidth(kde.factor*factor)
    p.line(x_grid,kde(x_grid),line_width=2,legend_label=label,color=color)
p.xaxis.axis_label='Mean radius'
p.yaxis.axis_label='Estimated density'
p.legend.location='top_right'
p.legend.click_policy='hide'
show(p)

# 9. Law of Large Numbers

Under suitable conditions,

$$
\bar X_n\xrightarrow{p}\mu.
$$

The key statement is probabilistic: with larger $n$, large deviations of the sample mean from the population mean become less likely.

A single running mean can be visually misleading, so we show **several random observation orderings**.

In [55]:
radius=df['mean_radius'].to_numpy()
ref=radius.mean()

p=figure(width=850,height=440,title='LLN: several running means from different random orderings', tools='pan,wheel_zoom,box_zoom,reset,save')
for seed,color in [(1,'indigo'),(2,'red'),(3,'green'),(4,'violet'),(5,'purple')]:
    rng=np.random.default_rng(seed)
    seq=rng.permutation(radius)
    running=np.cumsum(seq)/np.arange(1,len(seq)+1)
    p.line(np.arange(1,len(seq)+1),running,line_width=1.5,alpha=.75,legend_label=f'ordering {seed}',color=color)
p.line([1,len(radius)],[ref,ref],line_width=3,line_dash='dashed',legend_label=f'full-sample reference {ref:.3f}')
p.xaxis.axis_label='Number of observations accumulated'
p.yaxis.axis_label='Running mean'
p.legend.location='top_right'
p.legend.click_policy='hide'
show(p)

# 10. Central Limit Theorem

For iid data with finite variance,

$$
\frac{\sqrt n(\bar X-\mu)}{\sigma}\xrightarrow{d}N(0,1).
$$

Therefore

$$
SE(\bar X)\approx\frac{\sigma}{\sqrt n}.
$$

## Scenario B — How the sampling distribution changes with $n$

In [57]:
def resampled_means(values, sample_size, repetitions=4000, seed=42):
    rng=np.random.default_rng(seed+sample_size)
    values=np.asarray(values)
    out=np.empty(repetitions)
    for i in range(repetitions):
        out[i]=rng.choice(values,size=sample_size,replace=True).mean()
    return out

sample_sizes=[5,20,50,100]
colors=['red','yellow','indigo','blue']
clt_rows=[]

p=figure(width=850,height=450,title='Sampling distributions of the sample mean',tools='pan,wheel_zoom,box_zoom,reset,save')
for n_s,color in zip(sample_sizes,colors):
    m=resampled_means(radius,n_s)
    kde=stats.gaussian_kde(m)
    gx=np.linspace(m.min(),m.max(),400)
    p.line(gx,kde(gx),line_width=2,legend_label=f'n={n_s}',color=color)
    clt_rows.append({'n':n_s,'empirical_SE':m.std(ddof=1),'sigma_over_sqrt_n':radius.std(ddof=1)/np.sqrt(n_s),'skewness_of_means':stats.skew(m)})
p.xaxis.axis_label='Sample mean'
p.yaxis.axis_label='Estimated density'
p.legend.location='top_right'
p.legend.click_policy='hide'
show(p)
show_table(pd.DataFrame(clt_rows),'CLT diagnostics across sample sizes',height=230)

As $n$ increases:

1. the distribution contracts;
2. empirical standard error approaches $S/\sqrt n$;
3. the shape tends to become more Gaussian.

### Scenario C — CLT under strong skewness

The CLT does not say that $n=5$ is always enough.  For a more strongly skewed feature such as `worst_area`, convergence can require a larger sample.

In [58]:
skewed=df['worst_area'].to_numpy()
p=figure(width=850,height=450,title='CLT stress test using strongly skewed worst_area',tools='pan,wheel_zoom,box_zoom,reset,save')
rows=[]
for n_s in [5,20,50,100]:
    m=resampled_means(skewed,n_s,seed=100)
    kde=stats.gaussian_kde(m)
    gx=np.linspace(m.min(),m.max(),400)
    p.line(gx,kde(gx),line_width=2,legend_label=f'n={n_s}')
    rows.append({'n':n_s,'skewness_raw':stats.skew(skewed),'skewness_sample_means':stats.skew(m),'SE':m.std(ddof=1)})
p.xaxis.axis_label='Sample mean of worst_area'
p.yaxis.axis_label='Estimated density'
p.legend.location='top_right'
show(p)
show_table(pd.DataFrame(rows),'CLT convergence under skewness',height=230)

# 11. Method of Moments and Maximum Likelihood

Assume as a working model

$$
X\sim N(\mu,\sigma^2).
$$

The first two moments give

$$
\hat\mu_{MOM}=\bar X,
$$

$$
\hat\sigma^2_{MOM}=\frac1n\sum_i(X_i-\bar X)^2.
$$

For the normal model, these also equal the MLEs.  Notice that the MLE variance uses denominator $n$, whereas the unbiased sample variance uses $n-1$.

In [59]:
x=df['mean_radius'].to_numpy()
estimates=pd.DataFrame({
    'estimator':['MOM / MLE mean','MOM / MLE variance','Unbiased sample variance'],
    'value':[x.mean(),np.mean((x-x.mean())**2),x.var(ddof=1)]
})
show_table(estimates,'Normal-model estimators',height=210)

# 12. Estimator bias, variance and consistency

An estimator $\hat\theta$ is unbiased if

$$
E[\hat\theta]=\theta.
$$

Consistency means

$$
\hat\theta_n\xrightarrow{p}\theta.
$$

We simulate from a known population to separate mathematical truth from the unknown real-data population.

In [60]:
def estimator_sim(n,repetitions=5000,mu=10.0,sigma=3.0,seed=42):
    rng=np.random.default_rng(seed+n)
    est=np.empty(repetitions)
    for i in range(repetitions):
        est[i]=rng.normal(mu,sigma,size=n).mean()
    return {
        'n':n,
        'mean_estimator':est.mean(),
        'bias':est.mean()-mu,
        'variance':est.var(ddof=1),
        'theoretical_variance':sigma**2/n,
        'rmse':np.sqrt(np.mean((est-mu)**2)),
    }

est_df=pd.DataFrame([estimator_sim(n) for n in [5,10,20,50,100,500]])
show_table(est_df,'Sampling behaviour of the sample mean',height=280)
show(multi_line_plot(est_df,'n',{'variance':'empirical variance','theoretical_variance':'theoretical variance'},
                     'Estimator variance shrinks as n increases','Sample size n','Variance'))

# 13. Bootstrap: uncertainty without a convenient formula

The nonparametric bootstrap repeatedly resamples the observed data with replacement.

For $B$ replications and an $O(n)$ statistic, the direct algorithm costs approximately

$$
\Theta(Bn)
$$

time and $\Theta(B)$ additional storage for the resulting statistics.

In [61]:
@dataclass(frozen=True)
class BootstrapResult:
    estimates: np.ndarray

    @property
    def standard_error(self):
        return float(np.std(self.estimates,ddof=1))

    def percentile_ci(self,level=.95):
        a=1-level
        lo,hi=np.quantile(self.estimates,[a/2,1-a/2])
        return float(lo),float(hi)


class Bootstrapper:
    def __init__(self,n_boot=3000,seed=42):
        self.n_boot=n_boot
        self.rng=np.random.default_rng(seed)

    def one_sample(self,values,statistic=np.mean):
        values=np.asarray(values)
        out=np.empty(self.n_boot)
        for b in range(self.n_boot):
            out[b]=statistic(self.rng.choice(values,size=len(values),replace=True))
        return BootstrapResult(out)

    def two_sample_difference(self,a,b,statistic=np.mean):
        a=np.asarray(a); b=np.asarray(b)
        out=np.empty(self.n_boot)
        for i in range(self.n_boot):
            aa=self.rng.choice(a,size=len(a),replace=True)
            bb=self.rng.choice(b,size=len(b),replace=True)
            out[i]=statistic(aa)-statistic(bb)
        return BootstrapResult(out)

## Scenario D — Mean versus median bootstrap uncertainty

The mean is efficient under well-behaved symmetric distributions but sensitive to extremes.  The median is robust but can have larger sampling variance.

In [62]:
bs=Bootstrapper(3000,42)
mean_boot=bs.one_sample(df['mean_area'],np.mean)
median_boot=bs.one_sample(df['mean_area'],np.median)

boot_cmp=pd.DataFrame([
    {'statistic':'mean','observed':df['mean_area'].mean(),'SE':mean_boot.standard_error,'CI_low':mean_boot.percentile_ci()[0],'CI_high':mean_boot.percentile_ci()[1]},
    {'statistic':'median','observed':df['mean_area'].median(),'SE':median_boot.standard_error,'CI_low':median_boot.percentile_ci()[0],'CI_high':median_boot.percentile_ci()[1]},
])
show_table(boot_cmp,'Bootstrap mean versus median',height=210)

p=figure(width=840,height=430,title='Bootstrap distributions: mean versus median of mean_area',tools='pan,wheel_zoom,box_zoom,reset,save')
for arr,label in [(mean_boot.estimates,'mean'),(median_boot.estimates,'median')]:
    kde=stats.gaussian_kde(arr)
    gx=np.linspace(arr.min(),arr.max(),400)
    p.line(gx,kde(gx),line_width=2,legend_label=label)
p.legend.location='top_right'
p.xaxis.axis_label='Bootstrap statistic'
p.yaxis.axis_label='Estimated density'
show(p)

# 14. Classical and bootstrap confidence intervals

For a population mean with unknown variance, a classical Student-$t$ interval is

$$
\bar X\pm t_{1-\alpha/2,n-1}\frac{S}{\sqrt n}.
$$

A percentile bootstrap interval instead uses empirical quantiles of the bootstrap distribution.

In [63]:
def t_ci(values,level=.95):
    values=np.asarray(values)
    a=1-level
    m=values.mean(); se=stats.sem(values)
    crit=stats.t.ppf(1-a/2,df=len(values)-1)
    return float(m-crit*se),float(m+crit*se)

radius_boot=Bootstrapper(4000,7).one_sample(df['mean_radius'],np.mean)
ci_df=pd.DataFrame([
    {'method':'Student-t','lower':t_ci(df['mean_radius'])[0],'upper':t_ci(df['mean_radius'])[1]},
    {'method':'Bootstrap percentile','lower':radius_boot.percentile_ci()[0],'upper':radius_boot.percentile_ci()[1]},
])
show_table(ci_df,'95% confidence interval comparison for mean radius',height=200)

## Wilson interval for a Bernoulli proportion

The simple Wald interval can behave poorly near $0$ or $1$ and with small samples.  The Wilson interval is usually preferable.

In [64]:
wil_lo,wil_hi=proportion_confint(s,n,alpha=.05,method='wilson')
pd.DataFrame({'estimate':[p_hat],'wilson_lower':[wil_lo],'wilson_upper':[wil_hi]})

,estimate,wilson_lower,wilson_upper
0,0.3726,0.3338,0.4130


# 15. Comparing malignant and benign groups

Define

$$
\Delta=E[X\mid Y=1]-E[X\mid Y=0].
$$

We estimate the mean-radius difference and quantify uncertainty by bootstrapping the two groups separately.

In [65]:
a=df.loc[df.malignant==1,'mean_radius'].to_numpy()
b=df.loc[df.malignant==0,'mean_radius'].to_numpy()
obs=a.mean()-b.mean()
diff_boot=Bootstrapper(4000,11).two_sample_difference(a,b,np.mean)
lo,hi=diff_boot.percentile_ci()

show_table(pd.DataFrame([{'observed_difference':obs,'bootstrap_SE':diff_boot.standard_error,'CI_low':lo,'CI_high':hi}]),
           'Malignant − benign mean-radius difference',height=180)
show(histogram_plot(diff_boot.estimates,'Bootstrap distribution of mean difference','Malignant − benign mean radius',bins=40,density=True))

# 16. Hypothesis testing from multiple perspectives

We test

$$
H_0:\mu_M-\mu_B=0.
$$

Three methods answer related but not identical questions:

1. **Welch t-test** — parametric mean comparison without equal-variance assumption;
2. **Mann–Whitney U** — rank-based distributional comparison;
3. **Permutation test** — builds a null distribution by label shuffling.

In [66]:
welch=stats.ttest_ind(a,b,equal_var=False)
mw=stats.mannwhitneyu(a,b,alternative='two-sided')

def permutation_diff(values,labels,reps=5000,seed=42):
    rng=np.random.default_rng(seed)
    values=np.asarray(values); labels=np.asarray(labels)
    observed=values[labels==1].mean()-values[labels==0].mean()
    perm=np.empty(reps)
    for i in range(reps):
        lab=rng.permutation(labels)
        perm[i]=values[lab==1].mean()-values[lab==0].mean()
    p=(1+np.sum(np.abs(perm)>=abs(observed)))/(reps+1)
    return observed,perm,p

perm_obs,perm_stats,perm_p=permutation_diff(df.mean_radius,df.malignant)

tests=pd.DataFrame([
    {'test':'Welch t-test','statistic':welch.statistic,'p_value':welch.pvalue},
    {'test':'Mann-Whitney U','statistic':mw.statistic,'p_value':mw.pvalue},
    {'test':'Permutation difference','statistic':perm_obs,'p_value':perm_p},
])
show_table(tests,'Hypothesis-test comparison',height=220)
show(histogram_plot(perm_stats,'Permutation null distribution','Difference in means under H0',bins=45,density=True))

A small p-value is **not** $P(H_0\mid D)$.  It measures how surprising the observed statistic would be under the null model.

The permutation test is especially intuitive: if labels were exchangeable under $H_0$, repeatedly shuffling them shows what random group differences would look like.

# 17. Statistical significance versus effect size

With enough data, very small effects can become statistically significant.  We therefore pair p-values with an effect-size estimate.

Cohen's $d$ is

$$
d=\frac{\bar X_1-\bar X_2}{S_{pooled}}.
$$

In [67]:
def cohens_d(a,b):
    a=np.asarray(a); b=np.asarray(b)
    pooled=((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2)
    return (a.mean()-b.mean())/np.sqrt(pooled)

show_table(pd.DataFrame([{
    'difference_in_means':obs,
    'cohens_d':cohens_d(a,b),
    'bootstrap_CI_low':lo,
    'bootstrap_CI_high':hi,
    'Welch_p_value':welch.pvalue,
}]),'Effect size and inferential evidence',height=180)

# 18. Type-I error and power

Power is

$$
P(\text{reject }H_0\mid H_1).
$$

It increases with larger effects and larger samples.

## Scenario E — Power surface over sample size and effect size

In [68]:
def estimated_power(effect,n,reps=1200,alpha=.05,seed=0):
    rng=np.random.default_rng(seed+int(effect*1000)+n)
    reject=0
    for _ in range(reps):
        x=rng.normal(effect,1,n)
        z=rng.normal(0,1,n)
        reject += stats.ttest_ind(x,z,equal_var=False).pvalue < alpha
    return reject/reps

rows=[]
for eff in [0.0,0.2,0.35,0.5,0.8]:
    for n0 in [10,20,40,80,160]:
        rows.append({'effect_size':eff,'n_per_group':n0,'rejection_rate':estimated_power(eff,n0)})
power_df=pd.DataFrame(rows)

p=figure(width=820,height=440,title='Power / rejection-rate scenarios',tools='pan,wheel_zoom,box_zoom,reset,save')
for eff,g in power_df.groupby('effect_size'):
    p.line(g.n_per_group,g.rejection_rate,line_width=2,legend_label=f'effect={eff}')
    p.scatter(g.n_per_group,g.rejection_rate,size=6)
p.xaxis.axis_label='n per group'
p.yaxis.axis_label='Estimated rejection probability'
p.legend.location='bottom_right'
p.legend.click_policy='hide'
show(p)
show_table(power_df,'Power simulation grid',height=310)

When `effect_size = 0`, the rejection rate estimates the Type-I error rate and should hover near $\alpha=0.05$.

For nonzero effects, the same quantity estimates power.

# 19. Bayesian Beta–Bernoulli inference

Let

$$
p\sim\operatorname{Beta}(\alpha,\beta)
$$

and observe $s$ malignant cases among $n$ observations.

Then

$$
p\mid D\sim\operatorname{Beta}(\alpha+s,\beta+n-s).
$$

This conjugacy makes prior sensitivity easy to study.

In [70]:
def beta_posterior(alpha,beta,successes,total):
    return alpha+successes,beta+total-successes

priors=[(1,1,'uniform'),(2,2,'mildly centered'),(20,20,'strongly centered')]
p_axis=np.linspace(.15,.60,500)
p=figure(width=840,height=440,title='Prior sensitivity with the full dataset',tools='pan,wheel_zoom,box_zoom,reset,save')
rows=[]
for a0,b0,label in priors:
    ap,bp=beta_posterior(a0,b0,s,n)
    dens=stats.beta.pdf(p_axis,ap,bp)
    p.line(p_axis,dens,line_width=2,legend_label=label)
    ci=stats.beta.ppf([.025,.975],ap,bp)
    rows.append({'prior':label,'posterior_mean':ap/(ap+bp),'CI_low':ci[0],'CI_high':ci[1]})
p.xaxis.axis_label='p'
p.yaxis.axis_label='Posterior density'
p.legend.location='top_left'
show(p)
show_table(pd.DataFrame(rows),'Posterior summaries: full dataset',height=220)

## Scenario F — Priors matter much more with small data

Repeat the same analysis using only the first 20 observations.

In [71]:
small_y=df['malignant'].iloc[:20].to_numpy()
ss=int(small_y.sum()); nn=len(small_y)
p_axis=np.linspace(.05,.95,500)
p=figure(width=840,height=440,title='Prior sensitivity with only 20 observations',tools='pan,wheel_zoom,box_zoom,reset,save')
rows=[]
for a0,b0,label in priors:
    ap,bp=beta_posterior(a0,b0,ss,nn)
    p.line(p_axis,stats.beta.pdf(p_axis,ap,bp),line_width=2,legend_label=label)
    ci=stats.beta.ppf([.025,.975],ap,bp)
    rows.append({'prior':label,'posterior_mean':ap/(ap+bp),'CI_low':ci[0],'CI_high':ci[1]})
p.xaxis.axis_label='p'
p.yaxis.axis_label='Posterior density'
p.legend.location='top_right'
show(p)
show_table(pd.DataFrame(rows),'Posterior summaries: first 20 observations',height=220)

# 20. Linear regression as a statistical model

Model mean perimeter using mean radius:

$$
Y_i=\beta_0+\beta_1X_i+\varepsilon_i.
$$

Under Gaussian errors, OLS and MLE coincide for the regression coefficients.

In [28]:
X_ols=sm.add_constant(df[['mean_radius']])
y_ols=df['mean_perimeter']
ols=sm.OLS(y_ols,X_ols).fit()
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:         mean_perimeter   R-squared:                       0.996
Model:                            OLS   Adj. R-squared:                  0.996
Method:                 Least Squares   F-statistic:                 1.318e+05
Date:                Wed, 09 Sep 2026   Prob (F-statistic):               0.00
Time:                        15:16:28   Log-Likelihood:                -1070.9
No. Observations:                 569   AIC:                             2146.
Df Residuals:                     567   BIC:                             2155.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -5.2324      0.276    -18.960      

In [29]:
order=np.argsort(df.mean_radius.to_numpy())
xsort=df.mean_radius.to_numpy()[order]
yhat=ols.predict(X_ols).to_numpy()[order]

p=figure(width=820,height=440,title='Linear regression: perimeter ~ radius',tools='pan,wheel_zoom,box_zoom,reset,save,hover',
         tooltips=[('radius','$x{0.00}'),('perimeter','$y{0.00}')])
p.scatter(df.mean_radius,df.mean_perimeter,size=5,alpha=.45,legend_label='observations')
p.line(xsort,yhat,line_width=3,legend_label='OLS fit')
p.xaxis.axis_label='Mean radius'
p.yaxis.axis_label='Mean perimeter'
p.legend.location='top_left'
show(p)

## 20.1 Residual diagnostics

Residuals

$$
e_i=y_i-\hat y_i
$$

should be examined rather than assuming model adequacy from $R^2$ alone.

In [72]:
resid=ols.resid.to_numpy(); fitted=ols.fittedvalues.to_numpy()
p1=figure(width=600,height=400,title='Residuals versus fitted values',tools='pan,wheel_zoom,box_zoom,reset,save')
p1.scatter(fitted,resid,size=5,alpha=.5)
p1.line([fitted.min(),fitted.max()],[0,0],line_dash='dashed')
p1.xaxis.axis_label='Fitted value'; p1.yaxis.axis_label='Residual'

osm,osr=stats.probplot(resid,dist='norm',fit=False)
qfit=stats.linregress(osm,osr)
p2=figure(width=600,height=400,title='Normal Q-Q diagnostic',tools='pan,wheel_zoom,box_zoom,reset,save')
p2.scatter(osm,osr,size=5,alpha=.55)
qline=np.array([min(osm),max(osm)])
p2.line(qline,qfit.intercept+qfit.slope*qline,line_dash='dashed')
p2.xaxis.axis_label='Theoretical normal quantile'; p2.yaxis.axis_label='Ordered residual'
show(row(p1,p2))

# 21. Scenario G — Stress-testing a regression assumption

Real datasets do not provide controlled truth, so we create a small synthetic experiment to demonstrate heteroskedasticity.

Suppose

$$
Y=2+3X+\varepsilon
$$

but the error standard deviation grows with $X$.

The conditional mean is still linear, but constant-variance standard errors become questionable.

In [73]:
rng=np.random.default_rng(123)
xh=np.linspace(0,10,350)
sigma_h=.4+.35*xh
yh=2+3*xh+rng.normal(0,sigma_h)
Xh=sm.add_constant(xh)
model_h=sm.OLS(yh,Xh).fit()
robust_h=model_h.get_robustcov_results(cov_type='HC3')

coef_cmp=pd.DataFrame({
    'method':['classical OLS SE','HC3 robust SE'],
    'slope_estimate':[model_h.params[1],robust_h.params[1]],
    'slope_SE':[model_h.bse[1],robust_h.bse[1]],
    'slope_p_value':[model_h.pvalues[1],robust_h.pvalues[1]],
})
show_table(coef_cmp,'Classical versus heteroskedasticity-robust inference',height=200)

p=figure(width=820,height=430,title='Synthetic heteroskedasticity scenario',tools='pan,wheel_zoom,box_zoom,reset,save')
p.scatter(xh,yh,size=5,alpha=.45,legend_label='observations')
p.line(xh,model_h.predict(Xh),line_width=3,legend_label='OLS conditional mean')
p.xaxis.axis_label='X'; p.yaxis.axis_label='Y'; p.legend.location='top_left'
show(p)

The coefficient estimate can remain sensible while its **classical standard error** is not. Robust covariance estimators alter the uncertainty calculation without changing the fitted OLS coefficient.

# 22. Logistic regression: modelling a conditional probability

For binary $Y$,

$$
\log\frac{p_i}{1-p_i}=\beta_0+\beta^TX_i.
$$

The coefficients are estimated by maximizing Bernoulli likelihood.

In [32]:
logit_features=['mean_radius','mean_texture','mean_smoothness']
Xlog=sm.add_constant(df[logit_features])
ylog=df['malignant']
logit=sm.Logit(ylog,Xlog).fit(disp=False)
print(logit.summary())

coef=pd.DataFrame({
    'term':logit.params.index,
    'coefficient':logit.params.values,
    'odds_ratio':np.exp(logit.params.values),
    'std_error':logit.bse.values,
    'p_value':logit.pvalues.values,
})
show_table(coef,'Logistic-regression coefficients and odds ratios',height=240)

                           Logit Regression Results                           
Dep. Variable:              malignant   No. Observations:                  569
Model:                          Logit   Df Residuals:                      565
Method:                           MLE   Df Model:                            3
Date:                Wed, 09 Sep 2026   Pseudo R-squ.:                  0.7508
Time:                        15:16:28   Log-Likelihood:                -93.645
converged:                       True   LL-Null:                       -375.72
Covariance Type:            nonrobust   LLR p-value:                5.954e-122
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const             -42.0194      4.459     -9.423      0.000     -50.760     -33.279
mean_radius         1.3970      0.154      9.069      0.000       1.095       1.699
mean_texture        0.3806      

For predictor $X_j$, $e^{\beta_j}$ is the multiplicative change in the odds for a one-unit increase in $X_j$, holding the other included variables fixed.

Because the raw units differ greatly, odds ratios for one-unit changes should always be interpreted in the context of the variable's scale.

## 22.1 Visualizing $P(Y=1\mid X)$ using one predictor

In [74]:
X1=df[['mean_radius']]
y1=df['malignant']
pipe1=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000))])
pipe1.fit(X1,y1)
gx=np.linspace(X1.mean_radius.min(),X1.mean_radius.max(),450)
gp=pipe1.predict_proba(pd.DataFrame({'mean_radius':gx}))[:,1]

p=figure(width=840,height=440,title='Estimated P(malignant | mean radius)',tools='pan,wheel_zoom,box_zoom,reset,save')
p.scatter(df.mean_radius,df.malignant,size=4,alpha=.25,legend_label='observations')
p.line(gx,gp,line_width=3,legend_label='logistic probability')
p.xaxis.axis_label='Mean radius'; p.yaxis.axis_label='Estimated malignancy probability'
p.legend.location='top_left'
show(p)

# 23. Likelihood-ratio testing for nested logistic models

Compare an intercept-only model with the three-predictor model.

$$
G^2=2\left[\ell(\hat\theta_{full})-\ell(\hat\theta_{restricted})\right].
$$

Under regularity conditions and $H_0$,

$$
G^2\approx\chi^2_{df}.
$$

In [75]:
restricted=sm.Logit(ylog,np.ones((len(df),1))).fit(disp=False)
lr=2*(logit.llf-restricted.llf)
df_lr=int(logit.df_model-restricted.df_model)
p_lr=stats.chi2.sf(lr,df_lr)
show_table(pd.DataFrame([{
    'restricted_loglik':restricted.llf,
    'full_loglik':logit.llf,
    'LR_statistic':lr,
    'df':df_lr,
    'p_value':p_lr,
}]),'Likelihood-ratio test',height=180)

# 24. Inference is not the same as prediction

A coefficient can be statistically significant but contribute little to out-of-sample predictive performance.  We therefore move from inferential modelling to held-out evaluation.

In [76]:
predictors=['mean_radius','mean_texture','mean_perimeter','mean_area','mean_smoothness']
X=df[predictors]; y=df.malignant
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
clf=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000))])
clf.fit(Xtr,ytr)
prob=clf.predict_proba(Xte)[:,1]
pred=(prob>=.5).astype(int)
metrics=pd.DataFrame([{
    'accuracy':accuracy_score(yte,pred),
    'precision':precision_score(yte,pred),
    'recall':recall_score(yte,pred),
    'f1':f1_score(yte,pred),
    'roc_auc':roc_auc_score(yte,prob),
    'brier_score':brier_score_loss(yte,prob),
    'log_loss':log_loss(yte,prob),
}])
show_table(metrics,'Held-out predictive metrics',height=190)

## 24.1 ROC curve with Bokeh

ROC-AUC measures ranking/discrimination across all thresholds.

In [77]:
fpr,tpr,thr=roc_curve(yte,prob)
p=figure(width=650,height=500,title=f'ROC curve — AUC = {roc_auc_score(yte,prob):.3f}',tools='pan,wheel_zoom,box_zoom,reset,save')
p.line(fpr,tpr,line_width=3,legend_label='model')
p.line([0,1],[0,1],line_dash='dashed',legend_label='random ranking')
p.xaxis.axis_label='False positive rate'; p.yaxis.axis_label='True positive rate'
p.legend.location='bottom_right'
show(p)

## 24.2 Calibration

A discriminative model ranks cases correctly.  A calibrated model also produces probabilities whose numerical values correspond to observed frequencies.

For example, among observations assigned probability near 0.8, roughly 80% should be positive in a well-calibrated model.

In [78]:
def calibration_table(y_true,prob,n_bins=8):
    d=pd.DataFrame({'y':np.asarray(y_true),'p':np.asarray(prob)})
    # rank-based bins avoid empty bins with strongly concentrated probabilities
    d['bin']=pd.qcut(d['p'],q=min(n_bins,len(d)),duplicates='drop')
    out=(d.groupby('bin',observed=True)
          .agg(mean_predicted=('p','mean'),observed_rate=('y','mean'),count=('y','size'))
          .reset_index(drop=True))
    return out

cal=calibration_table(yte,prob,8)
p=figure(width=650,height=500,title='Calibration curve',tools='pan,wheel_zoom,box_zoom,reset,save')
p.line(cal.mean_predicted,cal.observed_rate,line_width=3,legend_label='model')
p.scatter(cal.mean_predicted,cal.observed_rate,size=8)
p.line([0,1],[0,1],line_dash='dashed',legend_label='perfect calibration')
p.xaxis.axis_label='Mean predicted probability'; p.yaxis.axis_label='Observed positive rate'
p.legend.location='top_left'
show(p)
show_table(cal,'Calibration bins',height=260)

# 25. Scenario H — Decision threshold trade-offs

The default threshold 0.5 is not a law of probability.  It is a decision rule.

Changing the threshold changes:

* sensitivity/recall;
* specificity;
* precision;
* F1;
* false-positive/false-negative trade-offs.

This distinction is critical in high-stakes classification.

In [79]:
thresholds=np.linspace(.05,.95,19)
rows=[]
for t in thresholds:
    pr=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(yte,pr,labels=[0,1]).ravel()
    rows.append({
        'threshold':t,
        'precision':precision_score(yte,pr,zero_division=0),
        'recall':recall_score(yte,pr,zero_division=0),
        'specificity':tn/(tn+fp),
        'f1':f1_score(yte,pr,zero_division=0),
    })
thr_df=pd.DataFrame(rows)
show(multi_line_plot(thr_df,'threshold',{
    'precision':'precision','recall':'recall','specificity':'specificity','f1':'F1'
},'Threshold sensitivity analysis','Probability threshold','Metric value'))
show_table(thr_df,'Threshold-performance table',height=320)

## 25.1 Confusion matrix at the conventional 0.5 threshold

In [80]:
cm=confusion_matrix(yte,pred,labels=[0,1])
cm_long=pd.DataFrame([
    {'actual':'benign','predicted':'benign','count':cm[0,0]},
    {'actual':'benign','predicted':'malignant','count':cm[0,1]},
    {'actual':'malignant','predicted':'benign','count':cm[1,0]},
    {'actual':'malignant','predicted':'malignant','count':cm[1,1]},
])
source=ColumnDataSource(cm_long)
mapper=LinearColorMapper(palette=Viridis256,low=cm_long['count'].min(),high=cm_long['count'].max())
p=figure(x_range=['benign','malignant'],y_range=['malignant','benign'],width=560,height=450,
         title='Confusion matrix at threshold 0.5',tools='hover,save,reset',
         tooltips=[('actual','@actual'),('predicted','@predicted'),('count','@count')])
p.rect(x='predicted',y='actual',width=1,height=1,source=source,fill_color={'field':'count','transform':mapper},line_color=None)
p.add_layout(ColorBar(color_mapper=mapper),'right')
show(p)

# 26. Scenario I — Does stronger evidence imply much better prediction?

We compare a one-feature logistic classifier against a richer five-feature model on the **same train/test split**.

This lets us distinguish:

* evidence that additional coefficients are nonzero;
* practical gain in held-out discrimination.

In [81]:
def fit_eval(cols):
    m=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000))])
    m.fit(Xtr[cols],ytr)
    pp=m.predict_proba(Xte[cols])[:,1]
    return {
        'features':'+'.join(cols),
        'n_features':len(cols),
        'ROC_AUC':roc_auc_score(yte,pp),
        'Brier':brier_score_loss(yte,pp),
        'LogLoss':log_loss(yte,pp),
    }

model_compare=pd.DataFrame([
    fit_eval(['mean_radius']),
    fit_eval(['mean_radius','mean_texture']),
    fit_eval(predictors),
])
show_table(model_compare,'Predictive gain from additional features',height=220)

This comparison is deliberately predictive rather than inferential.  A richer model can be statistically distinguishable from a simpler one yet produce only a modest change in ROC-AUC if the simpler feature already contains most of the ranking information.

# 27. Scenario J — Transformation of a skewed variable

Strong right skew can be reduced using

$$
Z=\log(1+X).
$$

Transformations can improve visualization, stabilize variance and make some model assumptions more plausible.  They also change the interpretation of coefficients, so they should not be applied mechanically.

In [82]:
raw_area=df['worst_area'].to_numpy()
log_area=np.log1p(raw_area)

p1=histogram_plot(raw_area,'Raw worst_area','worst_area',bins=30,density=True,width=600,height=400)
p2=histogram_plot(log_area,'log(1 + worst_area)','log1p(worst_area)',bins=30,density=True,width=600,height=400)
show(row(p1,p2))

show_table(pd.DataFrame([
    {'representation':'raw','skewness':stats.skew(raw_area),'kurtosis':stats.kurtosis(raw_area)},
    {'representation':'log1p','skewness':stats.skew(log_area),'kurtosis':stats.kurtosis(log_area)},
]),'Effect of a log transform on distribution shape',height=200)

# 28. A disciplined statistical workflow

A reliable ST5201X-style workflow is:

### 1. Define the population/question

What quantity do you actually want to learn?

### 2. Identify random variables and sampling assumptions

Are observations plausibly independent? Are they identically distributed? Was sampling biased?

### 3. Explore the data

Look at center, spread, skewness, dependence and unusual observations.

### 4. Choose a probability model or nonparametric strategy

Do not choose a distribution solely because it is mathematically convenient.

### 5. State the estimator

For example,

$$
\hat p=\bar Y.
$$

### 6. Quantify uncertainty

Use analytical standard errors, bootstrap, or posterior uncertainty as appropriate.

### 7. Separate estimation from testing

Report effect sizes and intervals, not just p-values.

### 8. Check assumptions and sensitivity

Change bandwidths, priors, thresholds, transformations or model forms and see whether conclusions are stable.

### 9. Separate inference from prediction

Coefficient significance and out-of-sample accuracy are not the same objective.

### 10. Communicate the decision context

A probability model is useful only when connected to the real decision being made.

# 29. Further scenarios and exercises

## Exercise 1 — Bootstrap robustness under contamination

Take `mean_area`, add five extreme synthetic observations, and compare how the bootstrap distributions of the mean and median change.

**Question:** Which statistic is more robust, and what price does it pay in ordinary uncontaminated data?

---

## Exercise 2 — Coverage simulation

Simulate repeated normal samples from a known population and construct 95% Student-$t$ intervals.

Estimate

$$
P(\mu\in CI).
$$

It should approach 0.95 if the procedure is correctly implemented.

---

## Exercise 3 — Type-I error under non-normality

Compare Welch t-test and permutation-test rejection rates when both groups are drawn from the same skewed distribution.

---

## Exercise 4 — Prior-data conflict

Use a strongly informative Beta prior centered far away from the observed malignancy rate.

Compare posteriors for $n=20$ and the full dataset.

**Question:** When does the data overwhelm the prior?

---

## Exercise 5 — Multicollinearity

Fit a logistic model with `mean_radius`, `mean_perimeter` and `mean_area` together.

Then standardize features and inspect coefficient standard errors.

**Question:** Why can individual coefficients become unstable while predictions remain strong?

---

## Exercise 6 — Cost-sensitive thresholds

Suppose a false negative costs ten times as much as a false positive.

Define

$$
\text{Cost}(t)=10FN(t)+FP(t)
$$

and choose the threshold minimizing this empirical cost on a validation set.

This exercise connects probability estimation to decision theory.

# 30. Computational complexity and engineering notes

| Operation | Approximate time | Additional memory | Comment |
|---|---:|---:|---|
| sample mean | $\Theta(n)$ | $\Theta(1)$ | one pass |
| ECDF sorting | $\Theta(n\log n)$ | $\Theta(n)$ | dominated by sorting |
| dense correlation matrix | $\Theta(np^2)$ | $\Theta(p^2)$ | expensive when $p$ is large |
| naive bootstrap | $\Theta(Bn)$ | $\Theta(B)$ here | replications are parallelizable |
| permutation test | $\Theta(Bn)$ | $\Theta(B)$ | also embarrassingly parallel |
| KDE evaluation | roughly $\Theta(nm)$ | depends on implementation | $m$ grid points |
| logistic regression | solver-dependent | solver-dependent | depends on $n,p$, conditioning and iterations |

### Production tips

* use a single seeded `numpy.random.Generator` for reproducibility;
* separate data preparation, estimation and evaluation;
* vectorize where it genuinely improves readability and memory use;
* parallelize independent bootstrap/permutation replications for large $B$;
* avoid materializing a full $(B,n)$ resample matrix unless memory is abundant;
* standardize predictors before numerical optimization when scales differ greatly;
* validate statistical assumptions independently from predictive metrics;
* cache expensive deterministic preprocessing when repeating experiments.

# 31. Final conceptual map

The deepest lesson of ST5201X is that the object you compute from one dataset is only one realization of a random procedure.

$$
\boxed{\text{Estimator itself is a random variable before the sample is observed.}}
$$

Once that becomes intuitive, the rest of the subject connects naturally:

* sampling distributions explain standard errors;
* standard errors lead to confidence intervals;
* null distributions lead to hypothesis tests;
* resampling approximates difficult sampling distributions;
* likelihood turns probability models into estimators;
* Bayesian updating turns likelihood and prior information into posterior uncertainty;
* regression extends unconditional probability to conditional modelling;
* held-out evaluation distinguishes explanatory/inferential success from predictive success.

In short:

$$
\text{Data science without uncertainty quantification}
\neq
\text{statistical reasoning}.
$$